In [ ]:
"""
Analysis2: Event-anchored case studies (4 named events)

 not considering 2018-2026 window, just a demo run 4 small windows
Step 6 Diagnostic: Sample size feasibility check for CVE temporal lag analysis.

PURPOSE
-------
Before building the full event-window / cross-correlation / construct-ratio
pipeline, we need to know whether per-event, per-window, per-construct cell
counts are large enough to support statistical testing (chi-square / Fisher's
exact) or whether some analyses need to be pooled (event-synchronized
averaging across all events) rather than run per-event.

INPUT
-----
bat_posts_results.csv with (at minimum) these columns -- adjust the
COLUMN MAPPING section below to match your actual column names:

    - a post timestamp/date column (created_utc, date, timestamp, etc.)
    - EX, EMO, COG, MD            -> binary YES/NO (1/0, True/False, "YES"/"NO")
    - bat_score                   -> composite 0-4 score
    - subreddit                   -> (optional but recommended)

OUTPUT
------
- Printed summary tables (console)
- A CSV: diagnostic_cell_counts.csv  (every event x window x construct cell count)
- A CSV: diagnostic_summary.csv      (flags which combos are underpowered)
"""

import pandas as pd
import numpy as np
from datetime import timedelta

# ============================================================
# COLUMN MAPPING -- EDIT THESE TO MATCH YOUR ACTUAL CSV HEADERS
# ============================================================
INPUT_CSV = "bat_posts_results_with_dates.csv"

COL_DATE = "created_utc"      # <-- change to your actual date/timestamp column name
COL_SUBREDDIT = "subreddit"   # <-- change if different, or set to None if absent
CONSTRUCT_COLS = {
    "EX": "EX",     # <-- change to your actual column name for Exhaustion YES/NO
    "EMO": "EMO",   # <-- Emotional Impairment
    "COG": "COG",   # <-- Cognitive Impairment
    "MD": "MD",     # <-- Mental Distance
}
COL_BAT_SCORE = "bat_score"

# Minimum cell size thresholds (rule of thumb for chi-square validity:
# expected cell counts >= 5; we use a stricter "comfortable" threshold of 30
# for "construct YES count" since these are rare events at 3-16% base rate)
MIN_COMFORTABLE = 30
MIN_MARGINAL = 10

# ============================================================
# EVENT DEFINITIONS (confirmed dates from Step 2)
# ============================================================
EVENTS = {
    "Log4Shell": "2021-12-09",
    "SolarWinds": "2020-12-08",
    "CrowdStrike": "2024-07-19",
    "MOVEit": "2023-05-31",
}

# Window definitions relative to disclosure date (in days)
WINDOWS = {
    "pre_baseline": (-56, -1),     # 8 weeks before
    "crisis": (0, 14),             # disclosure through +2 weeks
    "hangover": (15, 42),          # +2 to +6 weeks
    "late_tail": (43, 84),         # +6 to +12 weeks
}


def load_data(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df):,} rows from {path}")
    print(f"Columns found: {list(df.columns)}\n")

    # created_utc from Reddit data is commonly a Unix epoch (int/float),
    # not an ISO string. Try epoch first, fall back to generic parsing.
    raw = df[COL_DATE]
    if pd.api.types.is_numeric_dtype(raw):
        print(f"'{COL_DATE}' looks numeric -- treating as Unix epoch seconds.")
        df[COL_DATE] = pd.to_datetime(raw, unit="s", errors="coerce", utc=True)
    else:
        # Could be numeric-as-string ("1607472000") or an actual date string.
        numeric_like = pd.to_numeric(raw, errors="coerce")
        frac_numeric = numeric_like.notna().mean()
        if frac_numeric > 0.9:
            print(f"'{COL_DATE}' is string-typed but >90% numeric-looking -- "
                  f"treating as Unix epoch seconds.")
            df[COL_DATE] = pd.to_datetime(numeric_like, unit="s", errors="coerce", utc=True)
        else:
            print(f"'{COL_DATE}' treated as a generic date string.")
            df[COL_DATE] = pd.to_datetime(raw, errors="coerce", utc=True)

    n_bad_dates = df[COL_DATE].isna().sum()
    if n_bad_dates > 0:
        pct_bad = n_bad_dates / len(df) * 100
        print(f"WARNING: {n_bad_dates:,} rows ({pct_bad:.1f}%) had unparseable "
              f"dates and will be dropped.")
        if pct_bad > 20:
            print("  >> This is a LARGE proportion. Stop and inspect raw "
                  f"'{COL_DATE}' values before trusting any result below --")
            print("     this looks like the same class of silent failure "
                  "flagged earlier in the project (Phase 2B date mismatch).")
            print(f"  Sample raw values: {raw.dropna().astype(str).head(5).tolist()}")
    df = df.dropna(subset=[COL_DATE])
    df[COL_DATE] = df[COL_DATE].dt.tz_localize(None)

    # Normalize construct columns to 0/1
    for name, col in CONSTRUCT_COLS.items():
        if col not in df.columns:
            print(f"WARNING: construct column '{col}' not found in CSV.")
            continue
        df[col] = df[col].apply(_to_binary)

    print(f"\nDate range in corpus: {df[COL_DATE].min()} to {df[COL_DATE].max()}")
    print(f"Total usable rows after date parsing: {len(df):,}")

    # Quick distributional sanity check -- catches uneven scraping coverage
    # (e.g. the kind of issue that flagged the 2018-2022 mismatch before)
    print("\nPost counts by year:")
    print(df[COL_DATE].dt.year.value_counts().sort_index().to_string())
    print()
    return df


def _to_binary(val):
    if pd.isna(val):
        return 0
    if isinstance(val, (int, float)):
        return int(val != 0)
    s = str(val).strip().upper()
    return 1 if s in ("YES", "Y", "TRUE", "1") else 0


def build_cell_counts(df):
    rows = []
    for event_name, date_str in EVENTS.items():
        event_date = pd.Timestamp(date_str)
        for window_name, (start_offset, end_offset) in WINDOWS.items():
            win_start = event_date + timedelta(days=start_offset)
            win_end = event_date + timedelta(days=end_offset)
            mask = (df[COL_DATE] >= win_start) & (df[COL_DATE] <= win_end)
            sub = df.loc[mask]

            n_total = len(sub)
            row = {
                "event": event_name,
                "window": window_name,
                "window_start": win_start.date(),
                "window_end": win_end.date(),
                "n_total_posts": n_total,
            }

            for construct_name, col in CONSTRUCT_COLS.items():
                if col in df.columns:
                    n_yes = int(sub[col].sum()) if n_total > 0 else 0
                    pct = (n_yes / n_total * 100) if n_total > 0 else np.nan
                    row[f"{construct_name}_yes_n"] = n_yes
                    row[f"{construct_name}_yes_pct"] = round(pct, 2)

            if COL_BAT_SCORE in df.columns and n_total > 0:
                row["bat_score_mean"] = round(sub[COL_BAT_SCORE].mean(), 3)
                row["bat_score_ge2_n"] = int((sub[COL_BAT_SCORE] >= 2).sum())
            else:
                row["bat_score_mean"] = np.nan
                row["bat_score_ge2_n"] = np.nan

            rows.append(row)

    return pd.DataFrame(rows)


def flag_underpowered(cell_df):
    """For each event x window x construct, flag whether the YES count
    is comfortable, marginal, or underpowered for statistical testing."""
    flags = []
    construct_names = list(CONSTRUCT_COLS.keys())

    for _, row in cell_df.iterrows():
        for construct_name in construct_names:
            yes_col = f"{construct_name}_yes_n"
            if yes_col not in row:
                continue
            n_yes = row[yes_col]
            if pd.isna(n_yes):
                status = "missing_column"
            elif n_yes >= MIN_COMFORTABLE:
                status = "comfortable"
            elif n_yes >= MIN_MARGINAL:
                status = "marginal"
            else:
                status = "underpowered"

            flags.append({
                "event": row["event"],
                "window": row["window"],
                "construct": construct_name,
                "n_yes": n_yes,
                "n_total_posts": row["n_total_posts"],
                "status": status,
            })

    return pd.DataFrame(flags)


def print_summary(cell_df, flag_df):
    print("=" * 70)
    print("CELL COUNTS: posts per event x window")
    print("=" * 70)
    print(cell_df[["event", "window", "window_start", "window_end", "n_total_posts"]]
          .to_string(index=False))

    print("\n" + "=" * 70)
    print("CONSTRUCT YES COUNTS per event x window")
    print("=" * 70)
    yes_cols = ["event", "window"] + [f"{c}_yes_n" for c in CONSTRUCT_COLS if f"{c}_yes_n" in cell_df.columns]
    print(cell_df[yes_cols].to_string(index=False))

    print("\n" + "=" * 70)
    print(f"POWER FLAGS (comfortable >= {MIN_COMFORTABLE}, marginal >= {MIN_MARGINAL}, else underpowered)")
    print("=" * 70)
    status_counts = flag_df["status"].value_counts()
    print(status_counts.to_string())

    print("\n--- UNDERPOWERED CELLS (n_yes < {}) ---".format(MIN_MARGINAL))
    underpowered = flag_df[flag_df["status"] == "underpowered"]
    if len(underpowered) > 0:
        print(underpowered.to_string(index=False))
    else:
        print("None! All event x window x construct cells have n_yes >= "
              f"{MIN_MARGINAL}.")

    print("\n--- RECOMMENDATION ---")
    pct_underpowered = (flag_df["status"] == "underpowered").mean() * 100
    print(f"{pct_underpowered:.1f}% of event x window x construct cells are underpowered.")
    if pct_underpowered > 40:
        print(">> High proportion underpowered. Recommend treating per-event,")
        print("   per-construct chi-square tests as EXPLORATORY/illustrative only.")
        print("   Make event-synchronized averaging (pooled across all 4 events)")
        print("   your primary confirmatory analysis instead.")
    elif pct_underpowered > 15:
        print(">> Moderate proportion underpowered -- likely concentrated in")
        print("   rarer constructs (COG, MD) and/or shorter windows (crisis).")
        print("   Consider widening the crisis window or pooling MD+COG as")
        print("   'impairment' for per-event tests, while keeping all 4")
        print("   constructs separate in the pooled event-synchronized analysis.")
    else:
        print(">> Most cells are adequately powered. Per-event chi-square /")
        print("   Fisher's exact tests should be viable as planned.")


if __name__ == "__main__":
    df = load_data(INPUT_CSV)
    cell_df = build_cell_counts(df)
    flag_df = flag_underpowered(cell_df)

    print_summary(cell_df, flag_df)

    cell_df.to_csv("diagnostic_cell_counts.csv", index=False)
    flag_df.to_csv("diagnostic_summary.csv", index=False)
    print("\nSaved: diagnostic_cell_counts.csv, diagnostic_summary.csv")

Loaded 135,897 rows from bat_posts_results_with_dates.csv
Columns found: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'created_utc', 'subreddit', 'created_date', 'year', 'month', 'day_of_week', 'hour', 'year_month']

'created_utc' looks numeric -- treating as Unix epoch seconds.

Date range in corpus: 2018-01-01 05:54:28 to 2026-04-25 23:31:20
Total usable rows after date parsing: 135,897

Post counts by year:
created_utc
2018    13347
2019    14328
2020    14302
2021    13835
2022    15539
2023    17395
2024    20278
2025    20082
2026     6791

CELL COUNTS: posts per event x window
      event       window window_start window_end  n_total_posts
  Log4Shell pre_baseline   2021-10-14 2021-12-08           2262
  Log4Shell       crisis   2021-12-09 2021-12-23            542
  Log4Shell     hangover   2021-12-24 2022-01-20            929
  Log4

the output from above is a demo run - what does this output tell us: 


Coverage exists where you need it. CrowdStrike (2024) and MOVEit (2023) both have solid post counts (commit volume per window in the 500-3000 range), which answers the worry from before — your corpus does extend into 2023-2024, it's not truncated early.
There's a visible volume pattern already, before even looking at burnout rates. Notice crisis-window counts are consistently the lowest of the four windows for every single event (Log4Shell 542, SolarWinds 544, CrowdStrike 875, MOVEit 530), while pre_baseline and late_tail are 2-4x higher. That's interesting on its own — either people post less during acute crises (too busy firefighting to post, which is itself a testable burnout-adjacent claim), or there's some other artifact (e.g., your subreddit scrape sampling, Reddit's own ranking/visibility effects, or a structural quirk in how pre_baseline/late_tail windows are wider in days than crisis). Worth checking: crisis is 15 days, hangover is 28 days, late_tail is 42 days, pre_baseline is 56 days — so part of that volume difference is just window length. You'll want to normalize to a rate (posts per day) before drawing any conclusion about volume, not raw counts